# Imports and setup

In [1]:
import numpy as np

from itertools import combinations, product as cartesian_product
from scipy.fft import rfft, rfftfreq, irfft

import mcfs

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

%matplotlib widget

# Simple examples total field

## Direct vs iFFT comparison for $\xi$ 1D

### Parameters

In [ ]:
seed = 137 
N_samples = 16
Npix = 512
L_box = 100

delta_v_list = [0.0, L_box / 8]

In [ ]:
dv = L_box / Npix
v = np.arange(Npix) * dv

rng = np.random.default_rng(seed)

### toy field generation

In [ ]:
# ------------ generate gaussian tau field ------------ #

def pk_template(k, A, n, k0, k_cut):
    kk = np.maximum(k, 1e-12)
    return A * (kk / k0) ** n * np.exp(-(kk / k_cut) ** 2)

pk_func = lambda k: pk_template(k, A=1.0, n=2.0, k0=2.0 * np.pi / L_box, k_cut=30.0 * 2.0 * np.pi / L_box)

gen = mcfs.toy_fields.ShiftedGaussianTauGenerator1D(
    Npix=Npix, L_box=L_box, delta_v_list=delta_v_list, pk_func=pk_func, mean_tau=1.0, contrast=0.35,
)

tau_fields = gen.sample(N_samples=N_samples, seed=seed)

# ------------ generate shifted spike tau field ------------ #

n_pairs = 3
sigma_list = [L_box / 1000, L_box / 1000]
amplitude_list = [10.0, 10.0]
gen = mcfs.toy_fields.ShiftedSpikeTauGenerator1D(
    Npix=Npix, L_box=L_box, delta_v_list=delta_v_list, sigma_list=sigma_list, amplitude_list=amplitude_list, n_pairs=n_pairs,
)
tau_fields = gen.sample(N_samples=N_samples, seed=seed)

In [ ]:
x_fields = np.exp(-tau_fields)

x1 = x_fields[:, 0]
x2 = x_fields[:, 1]
x = np.prod(x_fields, axis=1)

x1_mean = np.mean(x1, axis=1, keepdims=True)
x2_mean = np.mean(x2, axis=1, keepdims=True)
x_mean = np.mean(x, axis=1, keepdims=True)

delta_x1 = x1 / x1_mean - 1.0
delta_x2 = x2 / x2_mean - 1.0
delta_x = x / x_mean - 1.0

### Example plot

In [ ]:
idx = rng.choice(N_samples, size=2, replace=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(figsize=(14, 6), nrows=2, ncols=1, sharex=True)

for j, i in enumerate(idx):
    c = colors[j % len(colors)]

    axes[0].plot(v / L_box, x[i],  lw=1.0, alpha=0.8, color=c, ls="-")
    axes[0].plot(v / L_box, x1[i], lw=3.0, alpha=0.8, color=c, ls="--")
    axes[0].plot(v / L_box, x2[i], lw=3.0, alpha=0.8, color=c, ls=":")

    axes[1].plot(v / L_box, delta_x[i],  lw=1.0, alpha=0.8, color=c, ls="-")
    axes[1].plot(v / L_box, delta_x1[i], lw=3.0, alpha=0.8, color=c, ls="--")
    axes[1].plot(v / L_box, delta_x2[i], lw=3.0, alpha=0.8, color=c, ls=":")

axes[0].set_ylabel(r"$F$", fontsize=22)
axes[0].tick_params(axis="both", labelsize=22)

axes[1].set_xlabel(r"$v / L_\mathrm{box}$", fontsize=22)
axes[1].set_ylabel(r"$\delta$", fontsize=22)
axes[1].tick_params(axis="both", labelsize=22)

# Legend for line styles
style_legend = [
    Line2D([0], [0], color="k", lw=2.0, ls="-",  label=r"$x = x_1 x_2$"),
    Line2D([0], [0], color="k", lw=2.0, ls="--", label=r"$x_1$"),
    Line2D([0], [0], color="k", lw=2.0, ls=":",  label=r"$x_2$"),
]

# Legend for selected sample indices
sample_legend = [
    Line2D([0], [0], color=colors[j % len(colors)], lw=3.0, ls="-", label=fr"sample #${i}$") for j, i in enumerate(idx)
]

leg1 = axes[0].legend(handles=style_legend, fontsize=22, loc="lower left")
axes[0].add_artist(leg1)
leg2 = axes[1].legend(handles=sample_legend, fontsize=22, loc="lower left")
axes[1].add_artist(leg2)

plt.tight_layout()
plt.show()

### Compute $P1D$

In [ ]:
k = 2.0 * np.pi * rfftfreq(Npix, d=dv)
k_norm = k * L_box / (2.0 * np.pi)

fk = rfft(delta_x, axis=1)
Pk_per_sample = (dv / Npix) * (fk * fk.conj())
Pk = np.mean(Pk_per_sample, axis=0).real

### Plot $P1D$

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# Total P1D
ax.plot(k_norm[1:], Pk[1:], lw=3.0, color="black")

# Fundamental mode and harmonics
for j, delta_v_0 in enumerate(delta_v_list):
    if delta_v_0 != 0.0:
        c = colors[j % len(colors)]
        k0_norm = L_box / delta_v_0   # Characteristic frequency from Delta v_0
        n_max_harmonic = int(np.floor((Npix / 2) / k0_norm))

        ax.axvline(k0_norm, color=c, lw=2.0, ls="--")
        for n in range(0, n_max_harmonic + 1):
            ax.axvline(n * k0_norm, color=c, lw=1.0, ls=":", alpha=0.7)

# Custom legend
legend_handles = [
    Line2D([0], [0], color="black", lw=4.0, ls="-",  label=r"$P_{\rm 1D}(k)$"),
    Line2D([0], [0], color="grey", lw=3.0, ls="--", label=r"$L_{\rm box}/\Delta v_0$"),
    Line2D([0], [0], color="grey", lw=2.0, ls=":",  label=r"harmonics"),
]

ax.set_yscale("log")
ax.set_xlabel(r"$kL_\mathrm{box}/(2\pi)$", fontsize=22)
ax.set_ylabel(r"$P_{\rm 1D}$", fontsize=22)
ax.legend(handles=legend_handles, fontsize=22)

plt.tight_layout()
plt.show()

### Compute $\xi$ 1D

In [ ]:
lags = (np.arange(Npix) - Npix // 2) * dv
lags_norm = lags / L_box

lags_per = np.arange(Npix) * dv
lags_per_norm = lags_per / L_box

# Xi from inverse FFT of the total P1D
xi_per_sample = irfft(Pk_per_sample, n=Npix, axis=1) / dv
xi = np.mean(xi_per_sample, axis=0).real
xi_plot = np.fft.fftshift(xi)

# Function to compute the direct periodic Xi1D
def Xi1D_direct_periodic(x):
    N, Npix = x.shape
    xi = np.empty((N, Npix), dtype=float)
    for lag in range(Npix):
        xi[:, lag] = np.mean(x * np.roll(x, -lag, axis=1), axis=1)
    return xi

# Direct periodic Xi
xi_direct_per_sample = Xi1D_direct_periodic(delta_x)
xi_direct = np.mean(xi_direct_per_sample, axis=0)

### Plot $\xi$ 1D

In [ ]:
fig = plt.figure(figsize=(14, 6.5))
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3.2, 1.2], hspace=0.06)

ax = fig.add_subplot(gs[0])
ax_res = fig.add_subplot(gs[1], sharex=ax)

# Main panel
ax.plot(lags_norm, xi_plot, lw=4.0, color="k", label=r"$x$: iRFFT")
ax.plot(lags_per_norm, xi_direct, lw=1.0, color="red", ls="-", label=r"$x$: Direct")

ax.axvline(0.0, color="k", lw=1.2, alpha=0.8, ls=":")
ax.axvline(0.5, color="k", lw=1.2, alpha=0.8, ls=":")

for j, delta_v_0 in enumerate(delta_v_list):
    if delta_v_0 != 0.0:
        c = colors[j % len(colors)]

        ax.axvline(delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")
        ax.axvline(1.0 - delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")
        ymin, ymax = ax.get_ylim()
        ytext = ymin + 0.92 * (ymax - ymin)
        ax.text(0.5, ytext, r"$\frac{L_\mathrm{box}}{2}$", ha="right", va="top", fontsize=22)
        ax.text(delta_v_0 / L_box, ytext, r"$\frac{\Delta v_0}{L_\mathrm{box}}$", ha="right", va="top", fontsize=22)
        ax.text(1.0 - delta_v_0 / L_box, ytext, r"$1-\frac{\Delta v_0}{L_\mathrm{box}}$", ha="left", va="top", fontsize=22)

        ax_res.axvline(delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")
        ax_res.axvline(1.0 - delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")

ax.set_ylabel(r"$\xi_{\rm 1D}$", fontsize=22)
ax.tick_params(axis="both", labelsize=22, labelbottom=False)
ax.legend(fontsize=22)

# Residual panel: only over common domain 0 <= lag < 1/2
mask_ifft = lags_norm >= 0.0
x_overlap = lags_norm[mask_ifft]
xi_ifft_overlap = xi_plot[mask_ifft]

xi_direct_overlap = xi_direct[:len(x_overlap)]
xi_floor = 1e-12 * np.max(np.abs(xi_ifft_overlap)) + 1e-15
xi_den = np.maximum(np.abs(xi_ifft_overlap), xi_floor)
xi_residual = (xi_direct_overlap - xi_ifft_overlap) / xi_den

ax_res.plot(x_overlap, xi_residual, lw=2.0, color="black")

ax_res.fill_between(x_overlap, -1e-13, 1e-13, color="gray", alpha=0.25, zorder=0)
ax_res.axhline(0.0, color="k", lw=1.2, alpha=0.8)
ax_res.axvline(0.0, color="k", lw=1.2, alpha=0.8, ls=":")
ax_res.axvline(0.5, color="k", lw=1.2, alpha=0.8, ls=":")

ax_res.set_xlabel(r"$\Delta v / L_\mathrm{box}$", fontsize=22)
ax_res.set_ylabel(r"$\Delta$", fontsize=22)

plt.tight_layout()
plt.show()

## Sliders

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# ------------------------------------------------------------
# Fixed parameters
# ------------------------------------------------------------
seed = 137
N_samples = 16
Npix = 512
L_box = 100.0

n_pairs = 3
amplitude_list = [10.0, 10.0]

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# ------------------------------------------------------------
# Main plotting function Sliders
# ------------------------------------------------------------
def plot_toy_model(delta_frac=1/8, sigma1_frac=1/1000, sigma2_frac=1/1000):
    delta_v_2 = L_box * delta_frac
    sigma1 = L_box * sigma1_frac
    sigma2 = L_box * sigma2_frac

    dv = L_box / Npix
    v = np.arange(Npix) * dv

    rng = np.random.default_rng(seed)

    # -------- generate shifted spike tau field -------- #
    delta_v_list = [0.0, delta_v_2]
    sigma_list = [sigma1, sigma2]

    gen = mcfs.toy_fields.ShiftedSpikeTauGenerator1D(
        Npix=Npix, L_box=L_box, delta_v_list=delta_v_list, sigma_list=sigma_list, amplitude_list=amplitude_list, n_pairs=n_pairs,
    )
    tau_fields = gen.sample(N_samples=N_samples, seed=seed)

    x_fields = np.exp(-tau_fields)

    x1 = x_fields[:, 0]
    x2 = x_fields[:, 1]
    x = np.prod(x_fields, axis=1)

    x1_mean = np.mean(x1, axis=1, keepdims=True)
    x2_mean = np.mean(x2, axis=1, keepdims=True)
    x_mean = np.mean(x, axis=1, keepdims=True)

    delta_x1 = x1 / x1_mean - 1.0
    delta_x2 = x2 / x2_mean - 1.0
    delta_x = x / x_mean - 1.0

    # ============================================================
    # Figure 1: sample fields
    # ============================================================
    idx = rng.choice(N_samples, size=2, replace=False)

    fig, ax = plt.subplots(figsize=(14, 4))

    for j, i in enumerate(idx):
        c = colors[j % len(colors)]

        ax.plot(v / L_box, x[i],  lw=1.0, alpha=0.8, color=c, ls="-")
        ax.plot(v / L_box, x1[i], lw=3.0, alpha=0.8, color=c, ls="--")
        ax.plot(v / L_box, x2[i], lw=3.0, alpha=0.8, color=c, ls=":")

    ax.set_ylabel(r"$F$", fontsize=22)
    ax.tick_params(axis="both", labelsize=22)

    ax.set_xlabel(r"$v / L_\mathrm{box}$", fontsize=22)
    ax.tick_params(axis="both", labelsize=22)

    style_legend = [
        Line2D([0], [0], color="k", lw=2.0, ls="-",  label=r"$x = x_1 x_2$"),
        Line2D([0], [0], color="k", lw=2.0, ls="--", label=r"$x_1$"),
        Line2D([0], [0], color="k", lw=2.0, ls=":",  label=r"$x_2$"),
    ]

    sample_legend = [
        Line2D([0], [0], color=colors[j % len(colors)], lw=3.0, ls="-", label=fr"sample #${i}$") for j, i in enumerate(idx)
    ]

    leg1 = ax.legend(handles=style_legend, fontsize=18, loc="lower left")
    ax.add_artist(leg1)
    leg2 = ax.legend(handles=sample_legend, fontsize=18, loc="lower right")
    ax.add_artist(leg2)

    fig.suptitle(
        fr"$\Delta v_2/L_\mathrm{{box}}={delta_frac:.4g}$, "
        fr"$\sigma_1/L_\mathrm{{box}}={sigma1_frac:.4g}$, "
        fr"$\sigma_2/L_\mathrm{{box}}={sigma2_frac:.4g}$",
        fontsize=18
    )
    plt.show()

    # ============================================================
    # Figure 2: P1D
    # ============================================================
    k = 2.0 * np.pi * rfftfreq(Npix, d=dv)
    k_norm = k * L_box / (2.0 * np.pi)

    fk = rfft(delta_x, axis=1)
    Pk_per_sample = (dv / Npix) * (fk * fk.conj())
    Pk = np.mean(Pk_per_sample, axis=0).real

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(k_norm[1:], Pk[1:], lw=3.0, color="black")

    for j, delta_v_0 in enumerate(delta_v_list):
        if delta_v_0 != 0.0:
            c = colors[j % len(colors)]
            k0_norm = L_box / delta_v_0   # Characteristic frequency from Delta v_0
            n_max_harmonic = int(np.floor((Npix / 2) / k0_norm))

            ax.axvline(k0_norm, color=c, lw=2.0, ls="--")
            for n in range(0, n_max_harmonic + 1):
                ax.axvline(n * k0_norm, color=c, lw=1.0, ls=":", alpha=0.7)

    legend_handles = [
        Line2D([0], [0], color="black", lw=3.0, ls="-",  label=r"$P_{\rm 1D}(k)$"),
        Line2D([0], [0], color="black", lw=2.0, ls="--", label=r"$L_{\rm box}/\Delta v_2$"),
        Line2D([0], [0], color="black", lw=1.2, ls=":",  label=r"harmonics"),
    ]

    ax.set_yscale("log")
    ax.set_xlabel(r"$kL_\mathrm{box}/(2\pi)$", fontsize=22)
    ax.set_ylabel(r"$P_{\rm 1D}$", fontsize=22)
    ax.tick_params(axis="both", labelsize=22)
    ax.legend(handles=legend_handles, fontsize=18)
    plt.show()

    # ============================================================
    # Figure 3: Xi1D
    # ============================================================
    lags = (np.arange(Npix) - Npix // 2) * dv
    lags_norm = lags / L_box

    # Xi from inverse FFT of the total P1D
    xi_per_sample = irfft(Pk_per_sample, n=Npix, axis=1) / dv
    xi = np.mean(xi_per_sample, axis=0).real
    xi_plot = np.fft.fftshift(xi)

    fig, ax = plt.subplots(figsize=(14, 4))

    ax.plot(lags_norm, xi_plot, lw=4.0, color="k")

    ax.axvline(0.0, color="k", lw=1.2, alpha=0.8, ls=":")
    ax.axvline(0.5, color="k", lw=1.2, alpha=0.8, ls=":")

    for j, delta_v_0 in enumerate(delta_v_list):
        if delta_v_0 != 0.0:
            c = colors[j % len(colors)]

            ax.axvline(delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")
            ax.axvline(1.0 - delta_v_0 / L_box, color=c, lw=1.2, alpha=0.8, ls="--")
            ymin, ymax = ax.get_ylim()
            ytext = ymin + 0.92 * (ymax - ymin)
            ax.text(0.5, ytext, r"$\frac{L_\mathrm{box}}{2}$", ha="right", va="top", fontsize=22)
            ax.text(delta_v_0 / L_box, ytext, r"$\frac{\Delta v_0}{L_\mathrm{box}}$", ha="right", va="top", fontsize=22)
            ax.text(1.0 - delta_v_0 / L_box, ytext, r"$1-\frac{\Delta v_0}{L_\mathrm{box}}$", ha="left", va="top", fontsize=22)

    ax.set_xlim(-0.01, 0.51)

    ymin, ymax = ax.get_ylim()
    ytext = ymin + 0.92 * (ymax - ymin)
    ax.text(0.5, ytext, r"$\frac{L_\mathrm{box}}{2}$", ha="right", va="top", fontsize=22)
    ax.text(delta_frac, ytext, r"$\frac{\Delta v_2}{L_\mathrm{box}}$", ha="right", va="top", fontsize=22)
    ax.text(1.0 - delta_frac, ytext, r"$1-\frac{\Delta v_2}{L_\mathrm{box}}$", ha="left", va="top", fontsize=22)

    ax.set_ylabel(r"$\xi_{\rm 1D}$", fontsize=22)
    ax.set_xlabel(r"$\Delta v / L_\mathrm{box}$", fontsize=22)
    ax.tick_params(axis="both", labelsize=22)

    plt.show()

In [ ]:
delta_slider = widgets.FloatSlider(
    value=1/8, min=0.0, max=0.48, step=1 / Npix, description=r"Δv/L", readout_format=".4f",
    continuous_update=False
)

sigma1_slider = widgets.FloatLogSlider(
    value=1/1000, base=10, min=-4.0, max=-2.0,      # 1e-2
    step=0.05, description=r"σ1/L", readout_format=".4f", continuous_update=False
)

sigma2_slider = widgets.FloatLogSlider(
    value=1/1000, base=10, min=-4.0, max=-2.0,
    step=0.05, description=r"σ2/L", readout_format=".4f", continuous_update=False,
)

ui = widgets.VBox([delta_slider, sigma1_slider, sigma2_slider])

out = widgets.interactive_output(
    plot_toy_model, {"delta_frac": delta_slider, "sigma1_frac": sigma1_slider, "sigma2_frac": sigma2_slider},
)

display(ui, out)

# Decomposition

## Small utilities

In [ ]:
def subset_key(subset):
    return "".join(subset)

def subset_order(key):
    return len(key)

def all_subset_keys(labels):
    keys = []
    members = {}
    for r in range(1, len(labels) + 1):
        for comb in combinations(labels, r):
            key = subset_key(comb)
            keys.append(key)
            members[key] = comb
    return keys, members

def pretty_subset(key):
    if len(key) == 1:
        return key
    return f"({key})"

def pretty_pair(pair):
    a, b = pair
    return rf"${pretty_subset(a)}{pretty_subset(b)}$"

def Xi1D_direct_periodic(a, b):
    N, Npix = a.shape
    xi = np.empty((N, Npix), dtype=float)

    for lag in range(Npix):
        xi[:, lag] = np.mean(a * np.roll(b, -lag, axis=1), axis=1)

    return xi

def color_dict(items, cmap_name="turbo"):
    cmap = plt.get_cmap(cmap_name)
    n = max(len(items), 1)
    return {item: cmap(i / max(n - 1, 1)) for i, item in enumerate(items)}

In [ ]:
# ================================================================
# Parameters
# ================================================================

seed = 137 
N_samples = 16
Npix = 512
L_box = 100

delta_v_list = [0.0, L_box/100] # [0.0, L_box/100, L_box/5]

# ================================================================
# Derived quantities
# ================================================================

dv = L_box / Npix
v = np.arange(Npix) * dv

rng = np.random.default_rng(seed)

# ================================================================
# toy data generation
# ================================================================

n_pairs = 2
sigma_list = [L_box / 1000, L_box / 1000]
amplitude_list = [10.0, 10.0, 10.0]
gen = mcfs.toy_fields.ShiftedSpikeTauGenerator1D(
    Npix=Npix, L_box=L_box, delta_v_list=delta_v_list, sigma_list=sigma_list, amplitude_list=amplitude_list, n_pairs=n_pairs,
)
tau_fields = gen.sample(N_samples=N_samples, seed=seed)

# -------------------- compute fields -------------------- #
fields = {}
labels = []
for ii in range(tau_fields.shape[1]):
    _label = str(ii + 1)
    labels.append(_label)
    fields[_label] = np.exp(-tau_fields[:, ii])
    
x = np.prod(list(fields.values()), axis=0)

subset_keys, subset_members = all_subset_keys(labels)
composite_keys = [key for key in subset_keys if subset_order(key) >= 2]

# -------------------- Means and fluctuation fields -------------------- #
means = {key: np.mean(field, axis=1, keepdims=True) for key, field in fields.items()}

delta_single = {key: fields[key] / means[key] - 1.0 for key in labels}

x_mean = np.mean(x, axis=1, keepdims=True)
delta_x = x / x_mean - 1.0

# ================================================================
# Composite overdensity fields
# ================================================================
delta_terms = {}

for key in subset_keys:
    members = subset_members[key]
    delta_terms[key] = np.prod(np.stack([delta_single[m] for m in members], axis=0), axis=0)

# ================================================================
# Mean coefficients: c_12, c_13, c_23, c_123, ...
# ================================================================
c_terms_per_sample = {key: np.mean(delta_terms[key], axis=1, keepdims=True) for key in composite_keys}

c_terms = {key: float(np.mean(c_terms_per_sample[key])) for key in composite_keys}

C_per_sample = np.zeros((N_samples, 1))
for key in composite_keys:
    C_per_sample += c_terms_per_sample[key]

C_total = float(np.mean(C_per_sample))

coefficients = dict(c_terms)
coefficients["C"] = C_total

weight_per_sample = 1.0 / (1.0 + C_per_sample) ** 2

print("Mean coefficients:")
for key, value in coefficients.items():
    print(f"  {key}: {value:.6e}")

delta_sum = np.zeros_like(delta_x)
for key in subset_keys:
    delta_sum += delta_terms[key]

delta_x_from_terms = (delta_sum - C_per_sample) / (1.0 + C_per_sample)
print("max|delta_x - delta_x_from_terms| =", np.max(np.abs(delta_x - delta_x_from_terms)))

In [ ]:
# ================================================================
# Plot example fields
# ================================================================
idx = rng.choice(x.shape[0], size=2, replace=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
linestyles = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]

fig, axes = plt.subplots(figsize=(14, 6), ncols=1, nrows=2, sharex=True)

for j, i in enumerate(idx):
    c = colors[j % len(colors)]
    axes[0].plot(v / L_box, x[i], lw=1.4, alpha=0.85, color=c, ls="-")
    axes[1].plot(v / L_box, delta_x[i], lw=1.4, alpha=0.85, color=c, ls="-")

    for m, label in enumerate(labels):
        ls = linestyles[m % len(linestyles)]
        axes[0].plot(v / L_box, fields[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)
        axes[1].plot(v / L_box, delta_single[label][i], lw=2.5, alpha=0.65, color=c, ls=ls)

axes[0].set_ylabel(r"$F$", fontsize=22)
axes[0].tick_params(axis="both", labelsize=22)

axes[1].set_xlabel(r"$v / L_\mathrm{box}$", fontsize=22)
axes[1].set_ylabel(r"$\delta$", fontsize=22)
axes[1].tick_params(axis="both", labelsize=22)

style_legend = [Line2D([0], [0], color="k", lw=2.0, ls="-", label=rf"$x=\prod_j\, x_j$")]

for m, label in enumerate(labels):
    ls = linestyles[m % len(linestyles)]
    style_legend.append(Line2D([0], [0], color="k", lw=2.0, ls=ls, label=rf"$x_{label}$"))

axes[0].legend(handles=style_legend, fontsize=18, loc="lower left")

sample_legend = [
    Line2D([0], [0], color=colors[j % len(colors)], lw=3.0, ls="-", label=fr"sample #${i}$") for j, i in enumerate(idx)
]
leg2 = axes[1].legend(handles=sample_legend, fontsize=18, loc="lower left")
axes[1].add_artist(leg2)

plt.show()

In [ ]:
# ================================================================
# Fourier transforms
# ================================================================
k = 2.0 * np.pi * rfftfreq(Npix, d=dv)
k_norm = k * L_box / (2.0 * np.pi)

f_total = rfft(delta_x, axis=1)

f_terms = {key: rfft(delta_terms[key], axis=1) for key in subset_keys}

prefactor = dv / Npix

# ================================================================
# Total P1D
# ================================================================
Pk_total_per_sample = prefactor * f_total * f_total.conj()
Pk_total = np.mean(Pk_total_per_sample, axis=0)

# ================================================================
# General P1D decomposition terms
# ================================================================
ordered_pairs = list(cartesian_product(subset_keys, subset_keys))

Pk_pair_per_sample = {}
Pk_pair = {}
Pk_pair_contrib = {}

for pair in ordered_pairs:
    a, b = pair

    Pk_ab_per_sample = prefactor * f_terms[a] * f_terms[b].conj()

    Pk_pair_per_sample[pair] = Pk_ab_per_sample
    Pk_pair[pair] = np.mean(Pk_ab_per_sample, axis=0)
    Pk_pair_contrib[pair] = np.mean(weight_per_sample * Pk_ab_per_sample, axis=0)

Pk_sum_per_sample = np.zeros_like(Pk_total, dtype=complex)
Pk_sum = np.zeros_like(Pk_total, dtype=complex)
for pair in ordered_pairs:
    Pk_sum_per_sample += Pk_pair_contrib[pair]
    Pk_sum += Pk_pair[pair]
Pk_sum *= (1.0 / (1.0 + C_total) ** 2)

In [ ]:
# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def pair_order(pair):
    # e.g. ('1','12') -> 1 + 2 = 3
    return len(pair[0]) + len(pair[1])

def is_auto_pair(pair):
    return pair[0] == pair[1]

def draw_harmonics(ax):
    for j, delta_v_0 in enumerate(delta_v_list):
        if delta_v_0 != 0.0:
            c = colors[j % len(colors)]
            k0_norm = L_box / delta_v_0
            n_max_harmonic = int(np.floor((Npix / 2) / k0_norm))

            ax.axvline(k0_norm, color=c, lw=2.0, ls="--")
            for n in range(0, n_max_harmonic + 1):
                ax.axvline(n * k0_norm, color=c, lw=1.0, ls=":", alpha=0.7)


# ============================================================
# 1) TOTAL FIGURE: total terms + residuals
# ============================================================
fig = plt.figure(figsize=(14, 6.5))
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3.3, 1.2], hspace=0.06)

ax = fig.add_subplot(gs[0])
ax_res = fig.add_subplot(gs[1], sharex=ax)

# Fundamental mode and harmonics
draw_harmonics(ax)

# Total / recovered terms
ax.plot(k_norm[1:], Pk_total.real[1:], lw=6.0, color="k")
ax.plot(k_norm[1:], Pk_sum.real[1:], lw=3.0, color="limegreen", ls="-")
ax.plot(k_norm[1:], Pk_sum_per_sample.real[1:], lw=1.5, color="red", ls="--")

# Legend for total terms
custom_legend = [
    Line2D([0], [0], color="black", lw=4.0, label=r"Tot."),
    Line2D([0], [0], color="limegreen", lw=2.4, label=r"Sum"),
    Line2D([0], [0], color="red", lw=2.4, ls="--", label=r"Sum (per sample)"),
]
leg = ax.legend(handles=custom_legend, fontsize=18, loc="upper right")
ax.add_artist(leg)

# Legend for harmonics
legend_handles = [
    Line2D([0], [0], color="grey", lw=3.0, ls="--", label=r"$L_{\rm box}/\Delta v_0$"),
    Line2D([0], [0], color="grey", lw=2.0, ls=":",  label=r"harmonics"),
]
leg = ax.legend(handles=legend_handles, fontsize=22, loc="lower right")
ax.add_artist(leg)

ax.tick_params(axis="both", labelsize=16, labelbottom=False)
ax.set_ylabel(r"$P_{\rm 1D}$", fontsize=22)
# ax.set_yscale("log")

# -------------------- Residuals -------------------- #
Pk_ref = Pk_total.real[1:]
Pk_floor = 1e-12 * np.max(np.abs(Pk_ref)) + 1e-15
Pk_den = np.maximum(np.abs(Pk_ref), Pk_floor)

Pk_residual = (Pk_sum.real[1:] - Pk_ref) / Pk_den
ax_res.plot(k_norm[1:], Pk_residual, lw=1.0, color="limegreen")

Pk_residual = (Pk_sum_per_sample.real[1:] - Pk_ref) / Pk_den
ax_res.plot(k_norm[1:], Pk_residual, lw=1.0, color="red", ls="--")

ax_res.axhline(0.0, color="k", lw=1.5, alpha=0.8)
# ax_res.axhspan(-0.001, 0.001, color="gray", alpha=0.2)

ax_res.tick_params(axis="both", labelsize=15)
ax_res.set_xlabel(r"$kL_\mathrm{box}/(2\pi)$", fontsize=22)
ax_res.set_ylabel(r"$\Delta$", fontsize=20)

plt.tight_layout()
plt.show()


# ============================================================
# 2) ONE FIGURE WITH ONE PANEL PER ORDER
# ============================================================
orders = sorted(set(pair_order(pair) for pair in ordered_pairs))
cmap = plt.get_cmap("tab10")

fig, axes = plt.subplots(
    nrows=len(orders),
    ncols=1,
    figsize=(14, 4.2 * len(orders)),
    sharex=True,
    sharey=False,
)

# If there is only one order, make axes iterable
if len(orders) == 1:
    axes = [axes]

for ax, order in zip(axes, orders):
    pairs_this_order = [pair for pair in ordered_pairs if pair_order(pair) == order]

    # Fundamental mode and harmonics
    draw_harmonics(ax)

    # Plot all terms of this order
    legend_terms = []
    for i, pair in enumerate(pairs_this_order):
        color = cmap(i % 10)
        ls = "-" if is_auto_pair(pair) else "--"
        lw = 2.5 if is_auto_pair(pair) else 2.0

        ax.plot(
            k_norm[1:],
            Pk_pair[pair].real[1:],
            lw=lw,
            ls=ls,
            color=color,
            alpha=0.95,
        )

        legend_terms.append(
            Line2D(
                [0], [0],
                color=color,
                lw=lw,
                ls=ls,
                label=pretty_pair(pair),
            )
        )

    # Legend with the terms of this order
    leg_terms = ax.legend(
        handles=legend_terms,
        fontsize=14,
        loc="upper left",
        ncol=2,
    )
    ax.add_artist(leg_terms)

    # Style legend
    leg_style = ax.legend(
        handles=[
            Line2D([0], [0], color="k", lw=2.5, ls="-",  label="auto"),
            Line2D([0], [0], color="k", lw=2.0, ls="--", label="non-auto"),
            Line2D([0], [0], color="grey", lw=2.0, ls="--", label=r"$L_{\rm box}/\Delta v_0$"),
            Line2D([0], [0], color="grey", lw=1.5, ls=":",  label="harmonics"),
        ],
        fontsize=14,
        loc="upper right",
    )
    ax.add_artist(leg_style)

    ax.set_title(fr"Order ${order}$ contributions", fontsize=18)
    ax.set_ylabel(r"$P_{\rm 1D}$", fontsize=18)
    ax.tick_params(axis="both", labelsize=14)
    # ax.set_yscale("log")

axes[-1].set_xlabel(r"$kL_\mathrm{box}/(2\pi)$", fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Xi1D from inverse FFT
# ================================================================
lags = (np.arange(Npix) - Npix // 2) * dv
lags_per = np.arange(Npix) * dv

xi_total_per_sample = irfft(Pk_total_per_sample, n=Npix, axis=1) / dv
xi_total = np.mean(xi_total_per_sample, axis=0)
xi_total_plot = np.fft.fftshift(xi_total.real)

xi_pair_per_sample = {}
xi_pair = {}
xi_pair_contrib = {}

for pair in ordered_pairs:
    xi_ab_per_sample = irfft(Pk_pair_per_sample[pair], n=Npix, axis=1) / dv

    xi_pair_per_sample[pair] = xi_ab_per_sample
    xi_pair[pair] = np.mean(xi_ab_per_sample, axis=0)
    xi_pair_contrib[pair] = np.mean(weight_per_sample * xi_ab_per_sample, axis=0)

xi_decomp = np.zeros_like(xi_total, dtype=float)
for pair in ordered_pairs:
    xi_decomp += xi_pair_contrib[pair].real

xi_const = np.mean(weight_per_sample[:, 0] * C_per_sample[:, 0] ** 2)
xi_decomp = xi_decomp - xi_const
xi_decomp_plot = np.fft.fftshift(xi_decomp)

In [ ]:
lags = (np.arange(Npix) - Npix // 2) * dv
lags_norm = lags / L_box

In [ ]:
# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def pair_order(pair):
    # e.g. ('1', '12') -> 1 + 2 = 3
    return len(pair[0]) + len(pair[1])

def is_auto_pair(pair):
    return pair[0] == pair[1]

def get_delta_v_difference_info(delta_v_list, L_box):
    """
    Return all unique pairwise separations between the delta_v of the fields,
    reduced to the principal periodic interval [0, L_box/2].
    """
    diff_info = []
    for i in range(len(delta_v_list)):
        for j in range(i + 1, len(delta_v_list)):
            d = abs(delta_v_list[j] - delta_v_list[i]) % L_box
            d = min(d, L_box - d)   # periodic shortest separation
            d_norm = d / L_box
            if d_norm > 0.0:
                diff_info.append((i + 1, j + 1, d_norm))
    return diff_info

def draw_delta_v_differences(ax, delta_v_list, L_box):
    """
    Draw vertical lines at the pairwise delta_v differences and return legend handles.
    """
    diff_info = get_delta_v_difference_info(delta_v_list, L_box)
    cmap = plt.get_cmap("tab10")
    handles = []

    for n, (i, j, d_norm) in enumerate(diff_info):
        c = cmap(n % 10)
        ax.axvline(d_norm, color=c, lw=1.5, ls="--", alpha=0.9)
        handles.append(
            Line2D(
                [0], [0],
                color=c,
                lw=1.8,
                ls="--",
                label=fr"$\Delta v_{{{i}{j}}}/L_\mathrm{{box}}$"
            )
        )
    return handles


# ============================================================
# 1) TOTAL FIGURE: total terms + residuals
# ============================================================
fig = plt.figure(figsize=(14, 7.0))
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3.5, 1.3], hspace=0.06)

ax = fig.add_subplot(gs[0])
ax_res = fig.add_subplot(gs[1], sharex=ax)

# Total and recovered decomposition
ax.plot(lags_norm, xi_total_plot, lw=4.0, color="black", label=r"total: iRFFT")
ax.plot(lags_norm, xi_decomp_plot, lw=2.4, color="red", ls="--", label=r"weighted sum $-C^2$")

# Reference vertical lines
ax.axvline(0.5, color="k", lw=1.5, alpha=0.9, ls=":")
ax.axvline(0.0, color="k", lw=1.5, alpha=0.9, ls=":")

# Pairwise delta_v differences
delta_handles = draw_delta_v_differences(ax, delta_v_list, L_box)

ymin, ymax = ax.get_ylim()
ytext = ymin + 0.92 * (ymax - ymin)
ax.text(0.5, ytext, r"$L_\mathrm{box}/2$", ha="center", va="top", fontsize=15)

ax.set_ylabel(r"$\xi_{\rm 1D}(\Delta v)$", fontsize=18)
ax.tick_params(axis="both", labelsize=16, labelbottom=False)

# Legends
leg_main = ax.legend(
    handles=[
        Line2D([0], [0], color="black", lw=4.0, ls="-", label=r"total: iRFFT"),
        Line2D([0], [0], color="red", lw=2.4, ls="--", label=r"weighted sum $-C^2$"),
    ],
    fontsize=16,
    loc="upper right"
)
ax.add_artist(leg_main)

if len(delta_handles) > 0:
    leg_delta = ax.legend(handles=delta_handles, fontsize=14, loc="upper left", ncol=2)
    ax.add_artist(leg_delta)

# ---------------------- Residuals ---------------------- #
xi_ref_centered = xi_total_plot
xi_floor_centered = 1e-12 * np.max(np.abs(xi_ref_centered)) + 1e-15
xi_den_centered = np.maximum(np.abs(xi_ref_centered), xi_floor_centered)

xi_residual_decomp_vs_ifft = (xi_decomp_plot - xi_ref_centered) / xi_den_centered

ax_res.axhline(0.0, color="k", lw=1.2, alpha=0.8)
ax_res.plot(lags_norm, xi_residual_decomp_vs_ifft, lw=1.0, color="limegreen", ls="-")

ax_res.axvline(0.5, color="k", lw=1.2, alpha=0.7, ls=":")
ax_res.axvline(0.0, color="k", lw=1.2, alpha=0.7, ls=":")

# same delta_v-difference markers in residual panel
_ = draw_delta_v_differences(ax_res, delta_v_list, L_box)

ax_res.set_xlabel(r"$\Delta v / L_\mathrm{box}$", fontsize=18)
ax_res.set_ylabel(r"rel. diff.", fontsize=14)
ax_res.tick_params(axis="both", labelsize=15)

ax.set_xlim(-0.01, 0.51)

plt.tight_layout()
plt.show()


# ============================================================
# 2) ONE MULTI-PANEL FIGURE: one panel per order
# ============================================================
orders = sorted(set(pair_order(pair) for pair in ordered_pairs))
cmap = plt.get_cmap("tab10")

fig, axes = plt.subplots(
    nrows=len(orders),
    ncols=1,
    figsize=(14, 4.2 * len(orders)),
    sharex=True,
    sharey=False,
)

# Make sure axes is always iterable
if len(orders) == 1:
    axes = [axes]

for ax, order in zip(axes, orders):
    pairs_this_order = [pair for pair in ordered_pairs if pair_order(pair) == order]

    legend_terms = []
    for i, pair in enumerate(pairs_this_order):
        color = cmap(i % 10)
        ls = "-" if is_auto_pair(pair) else "--"
        lw = 2.5 if is_auto_pair(pair) else 2.0

        xi_this = np.fft.fftshift(xi_pair[pair].real)

        ax.plot(
            lags_norm,
            xi_this,
            lw=lw,
            ls=ls,
            color=color,
            alpha=0.95,
        )

        legend_terms.append(
            Line2D(
                [0], [0],
                color=color,
                lw=lw,
                ls=ls,
                label=pretty_pair(pair),
            )
        )

    # Reference vertical lines
    ax.axvline(0.5, color="k", lw=1.5, alpha=0.9, ls=":")
    ax.axvline(0.0, color="k", lw=1.5, alpha=0.9, ls=":")

    # Pairwise delta_v-difference lines
    delta_handles = draw_delta_v_differences(ax, delta_v_list, L_box)

    ymin, ymax = ax.get_ylim()
    ytext = ymin + 0.92 * (ymax - ymin)
    ax.text(0.5, ytext, r"$L_\mathrm{box}/2$", ha="center", va="top", fontsize=15)

    # Per-panel legend for terms
    leg_terms = ax.legend(
        handles=legend_terms,
        fontsize=12,
        loc="upper left",
        ncol=2
    )
    ax.add_artist(leg_terms)

    # Put the style legend only in the first panel
    if ax is axes[0]:
        leg_style = ax.legend(
            handles=[
                Line2D([0], [0], color="k", lw=2.5, ls="-",  label="auto"),
                Line2D([0], [0], color="k", lw=2.0, ls="--", label="non-auto"),
            ],
            fontsize=12,
            loc="upper right",
        )
        ax.add_artist(leg_style)

        if len(delta_handles) > 0:
            leg_delta = ax.legend(
                handles=delta_handles,
                fontsize=11,
                loc="lower right",
                ncol=2,
            )
            ax.add_artist(leg_delta)

    ax.set_title(fr"Order ${order}$ contributions", fontsize=18)
    ax.set_ylabel(r"$\xi_{\rm 1D}(\Delta v)$", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_xlim(-0.01, 0.51)

axes[-1].set_xlabel(r"$\Delta v / L_\mathrm{box}$", fontsize=18)

plt.tight_layout()
plt.show()